# Integrate growth data into the recoding AnnData

Takes the `recoding_landscape.h5ad` that `AGGREGATE_ANNDATA` publishes under
`${alignment.outdir}/recoding/anndata/` and joins the AMiGA growth summaries
(`${outdir}/amiga/summary/*_summary.txt`) onto `obs`, one growth row per clone.

Pipeline of this notebook:

1. **Config** — paths + the sequencing-plate → growth-plate mapping (the only part
   that changes per run).
2. **Assign** each clone a `plate_name` from its sample/clone number.
3. **Join** the AMiGA summaries onto `obs`, index-aligned (never a bare
   `obs.merge`, which reorders/drops rows and silently desynchronises `obs` from `X`).
4. **Derive** per-clone recoding metrics (`recoded_binary` layer, `recoded_total`,
   `recoded_fraction`, `mean_depth`) and filter.
5. **Summarise** growth per codon into `var` (mean/median/std/min/max over the
   clones recoded at that codon).
6. **Plots** + write `recoding_growth.h5ad`.

In [ ]:
import os
import re
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ad.settings.allow_write_nullable_strings = True
sns.set_theme(style='whitegrid')

## 1. Config

`RESULTS_DIR` is the pipeline's `alignment.outdir`. Everything else hangs off it.

`PLATE_RULES` maps clones to growth plates. Each rule matches on the `S<n>` number
from the FASTQ name (`sample_nr`) and, optionally, on the clone number parsed from
`_c<n>_` in the BAM name — needed when several 96-sample sequencing plates reuse the
same `S1..S96` numbering. Set `clone_nr: None` to match on `sample_nr` alone.

In [ ]:
RESULTS_DIR  = '/Users/cristian.soitu/Data/260410_VH02427_20_AAHYTYLM5/results'
ANNDATA_DIR  = os.path.join(RESULTS_DIR, 'recoding', 'anndata')
AMIGA_DIR    = os.path.join(RESULTS_DIR, 'amiga', 'summary')

IN_H5AD  = os.path.join(ANNDATA_DIR, 'recoding_landscape.h5ad')
OUT_H5AD = os.path.join(ANNDATA_DIR, 'recoding_growth.h5ad')

# growth_plate == the AMiGA Plate_ID == '<growth_plate>_summary.txt' in AMIGA_DIR.
# Ranges are inclusive. clone_nr=None -> match on sample_nr only.
PLATE_RULES = [
    {'growth_plate': '260403_MSB10mapping_d1_1',    'sample_nr': (1, 96),    'clone_nr': (1, 96)},
    {'growth_plate': '260403_MSB10mapping_d1_2',    'sample_nr': (97, 192),  'clone_nr': (97, 192)},
    {'growth_plate': '260404_B10MAP_D2_1_2_3_4_1',  'sample_nr': (193, 288), 'clone_nr': (1, 96)},
    {'growth_plate': '260404_B10MAP_D2_1_2_3_4_2',  'sample_nr': (1, 96),    'clone_nr': (97, 192)},
]

# Filters / thresholds
MIXED_TRACE_THRESHOLD = 0.85   # call a codon recoded above this recoding fraction
DEPTH_THRESHOLD       = 10     # drop a clone below this mean depth
TD_THRESHOLD_MIN      = 300    # drop a clone slower than this doubling time (minutes)
OD_THRESHOLD          = None   # set a float to drop clones below this OD_Max

print('\n'.join(f for f in sorted(os.listdir(AMIGA_DIR)) if f.endswith('_summary.txt')))

## 2. Load the recoding AnnData

In [ ]:
adata = ad.read_h5ad(IN_H5AD)
print(adata)
adata.obs.head()

## 3. Assign a growth plate to each clone

`clone_nr` comes from `_c<n>_` in `BAM_name` and is `NaN` when the names carry no
such token; a rule with `clone_nr: None` then still matches. Rules are tried in
order, first match wins.

In [ ]:
def parse_clone_nr(bam_name):
    m = re.search(r'_c(\d+)(?:_|$)', str(bam_name))
    return int(m.group(1)) if m else np.nan


def parse_sample_nr(sample_id):
    m = re.search(r'(\d+)$', str(sample_id))
    return int(m.group(1)) if m else np.nan


def in_range(value, rng):
    if rng is None:            # rule does not constrain this field
        return True
    if not np.isfinite(value): # name carries no such number -> cannot match
        return False
    return rng[0] <= value <= rng[1]


def assign_plate(sample_nr, clone_nr, rules=PLATE_RULES):
    for rule in rules:
        if in_range(sample_nr, rule['sample_nr']) and in_range(clone_nr, rule.get('clone_nr')):
            return rule['growth_plate']
    return None


adata.obs['clone_nr']  = adata.obs['BAM_name'].map(parse_clone_nr)
adata.obs['sample_nr'] = adata.obs['sample_ID'].map(parse_sample_nr)
adata.obs['plate_name'] = [
    assign_plate(s, c) for s, c in zip(adata.obs['sample_nr'], adata.obs['clone_nr'])
]

n_unassigned = adata.obs['plate_name'].isna().sum()
print(adata.obs['plate_name'].value_counts(dropna=False))
print(f'\nunassigned clones: {n_unassigned}')
if n_unassigned:
    display(adata.obs.loc[adata.obs['plate_name'].isna(),
                          ['BAM_name', 'sample_ID', 'clone_nr', 'well_ID']].head(10))

Each (plate, well) pair should occur once — a duplicate means two clones claim the
same growth curve and the join below would fan out.

In [ ]:
dups = (adata.obs.dropna(subset=['plate_name'])
        .groupby(['plate_name', 'well_ID'], observed=True)
        .size().loc[lambda s: s > 1])
print(f'duplicated (plate_name, well_ID) pairs: {len(dups)}')
dups.head(20)

## 4. Read the AMiGA growth summaries

In [ ]:
growth = pd.concat(
    [pd.read_csv(os.path.join(AMIGA_DIR, f'{r["growth_plate"]}_summary.txt'), sep='\t')
     for r in PLATE_RULES],
    ignore_index=True,
)

# AMiGA's own Sample_ID is a per-plate row counter, not our sequencing sample; drop
# it so it cannot collide with obs['sample_ID'].
growth = growth.drop(columns=[c for c in ['Sample_ID'] if c in growth.columns])

# Plate_ID inside the file is the source of truth for what we just read.
print(growth['Plate_ID'].value_counts())
print(f'\ngrowth rows: {len(growth)}')
growth.head()

## 5. Join growth onto `obs`

Index-aligned left join: build the growth block against `adata.obs_names`, then
concatenate. `obs` keeps its order and length, so it stays row-for-row consistent
with `X` and the layers. Clones with no growth row get NaNs, and are dropped
explicitly afterwards.

In [ ]:
growth_keyed = growth.set_index(['Plate_ID', 'Well'])
if growth_keyed.index.has_duplicates:
    raise ValueError('duplicate (Plate_ID, Well) rows in the AMiGA summaries')

keys = pd.MultiIndex.from_arrays([adata.obs['plate_name'].to_numpy(),
                                  adata.obs['well_ID'].astype(str).to_numpy()])
growth_block = growth_keyed.reindex(keys)
growth_block.index = adata.obs_names

overlap = growth_block.columns.intersection(adata.obs.columns)
if len(overlap):
    print(f'overwriting existing obs columns: {list(overlap)}')
    adata.obs = adata.obs.drop(columns=overlap)

adata.obs = pd.concat([adata.obs, growth_block], axis=1)

matched = adata.obs['td'].notna()
print(f'clones with growth data: {matched.sum()} / {adata.n_obs}')
adata = adata[matched].copy()
adata.obs.head()

## 6. Derived per-clone metrics and filtering

- `td_min` — doubling time in minutes (AMiGA reports `td` in hours).
- `OD_death` — `death_lin`, renamed for readability.
- `recoded_binary` — per-codon recoding call at `MIXED_TRACE_THRESHOLD`.
- `recoded_total` / `recoded_fraction` — codons recoded per clone.
- `mean_depth` — mean sequencing depth over all codons.

In [ ]:
adata.obs['td_min'] = adata.obs['td'] * 60
adata.obs = adata.obs.rename(columns={'death_lin': 'OD_death'})

adata.layers['recoded_binary'] = (adata.X > MIXED_TRACE_THRESHOLD).astype(int)
adata.obs['recoded_total']    = adata.layers['recoded_binary'].sum(axis=1)
adata.obs['recoded_fraction'] = adata.obs['recoded_total'] / adata.n_vars
adata.obs['mean_depth']       = adata.layers['depth'].mean(axis=1)

n0 = adata.n_obs
adata = adata[adata.obs['mean_depth'] >= DEPTH_THRESHOLD].copy()
print(f'mean depth >= {DEPTH_THRESHOLD}: {adata.n_obs} / {n0}')

n0 = adata.n_obs
adata = adata[adata.obs['td_min'] <= TD_THRESHOLD_MIN].copy()
print(f'td <= {TD_THRESHOLD_MIN} min:    {adata.n_obs} / {n0}')

if OD_THRESHOLD is not None:
    n0 = adata.n_obs
    adata = adata[adata.obs['OD_Max'] > OD_THRESHOLD].copy()
    print(f'OD_Max > {OD_THRESHOLD}:        {adata.n_obs} / {n0}')

adata.obs[['td_min', 'OD_Max', 'OD_death', 'recoded_fraction', 'mean_depth']].describe()

## 7. Summarise growth per codon into `var`

For each codon, the distribution of a growth metric across the clones recoded at
that codon. Codons that no surviving clone recodes come out as NaN (rather than
raising), and `n_recoded` records how many clones each statistic rests on.

In [ ]:
GROWTH_METRICS = ['td_min', 'OD_Max', 'OD_death']
STATS = ['min', 'max', 'mean', 'median', 'std']

mask = adata.layers['recoded_binary'] > 0                 # (n_obs, n_vars)
adata.var['n_recoded'] = mask.sum(axis=0)

with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)       # all-NaN columns
    for metric in GROWTH_METRICS:
        values = adata.obs[metric].to_numpy(dtype=float)[:, None]
        masked = np.where(mask, values, np.nan)
        for stat in STATS:
            adata.var[f'{metric}_{stat}'] = getattr(np, f'nan{stat}')(masked, axis=0)

print(f'codons with no recoded clone: {(adata.var["n_recoded"] == 0).sum()} / {adata.n_vars}')
adata.var.head()

## 8. QC plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.scatterplot(x=adata.obs['recoded_fraction'], y=adata.obs['OD_Max'], s=12, alpha=.6, ax=axes[0])
axes[0].set(xlabel='Recoded fraction', ylabel='Max OD', title='Max OD vs recoded fraction')

sns.scatterplot(x=adata.obs['recoded_fraction'], y=adata.obs['td_min'], s=12, alpha=.6, ax=axes[1])
axes[1].set(xlabel='Recoded fraction', ylabel='Doubling time (min)', title='Doubling time vs recoded fraction')

sns.scatterplot(x=adata.obs['OD_Max'], y=adata.obs['td_min'], s=12, alpha=.6, ax=axes[2])
axes[2].set(xlabel='Max OD', ylabel='Doubling time (min)', title='Doubling time vs max OD')

fig.tight_layout()
plt.show()

In [ ]:
pos = adata.var['position'].to_numpy()
mean, std = adata.var['td_min_mean'].to_numpy(), adata.var['td_min_std'].to_numpy()

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(pos, mean, lw=.8, label='mean doubling time')
ax.fill_between(pos, mean - std, mean + std, alpha=.2, label='± 1 SD')
ax.set(xlabel='Position (bp)', ylabel='Doubling time (min)',
       title='Per-codon doubling time of recoded clones along the genome')
ax.set_xticks(np.arange(pos.min(), pos.max(), 200_000))
ax.set_ylim(bottom=0)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(pos, adata.var['OD_Max_mean'].to_numpy(), lw=.8, color='tab:orange', label='mean max OD')
ax.set(xlabel='Position (bp)', ylabel='Max OD',
       title='Per-codon max OD of recoded clones along the genome')
ax.set_xticks(np.arange(pos.min(), pos.max(), 200_000))
ax.set_ylim(bottom=0)
ax.legend()
fig.tight_layout()
plt.show()

## 9. Write out

`recoding_growth.h5ad` sits next to the pipeline's `recoding_landscape.h5ad`, plus
flat `obs` / `var` tables for anything that would rather read CSV.

In [ ]:
adata.write_h5ad(OUT_H5AD)
adata.obs.to_csv(os.path.join(ANNDATA_DIR, 'recoding_growth_obs.csv'))
adata.var.to_csv(os.path.join(ANNDATA_DIR, 'recoding_growth_var.csv'))
print(f'wrote {OUT_H5AD}  shape={adata.shape}')
adata